In [1]:
!mkdir -p ~/work/transformer_chatbot/data/spa-eng

In [2]:
!python3 -m pip install --upgrade pip
!python3 -m pip install konlpy
!pip install mecab-python3
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 54.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [konlpy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 11.9 MB/s  0:00:00
Install mecab-ko
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1381k  100 1381k    0     0  1240k      0  0:00:01  0:00:01 --:--:-- 3892k
mecab-0.996-ko-0.9.2/
mecab-0.996-ko-0.9.2/example/
mecab-0.996-ko-0.9.2/example/example.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.cpp
mecab-0.996-ko-0.9.2/example/example_lattice.c
mecab-0.996-ko-0.9.2/example/example.c
mecab-0.

In [3]:
!pip install sentencepiece nltk

In [4]:
import numpy as np
import pandas as pd
import torch
import sentencepiece as spm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

import re
import os
import random
import math

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print(torch.__version__)

2.10.0+cu128


In [5]:
import urllib.request
import zipfile

zip_filename = "spa-eng.zip"
zip_url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"

urllib.request.urlretrieve(zip_url, zip_filename)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(os.path.dirname(zip_filename))

In [6]:
extracted_folder = "./spa-eng"
file_path = os.path.join(extracted_folder, "spa.txt")

with open(file_path, "r") as f:
    spa_eng_sentences = f.read().splitlines()

spa_eng_sentences = list(set(spa_eng_sentences))
total_sentence_count = len(spa_eng_sentences)
print("Example:", total_sentence_count)

for sen in spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Example: 118964
>> Tom held up his hands.	Tom levantó sus manos.
>> I've done a lot of very bad things.	He hecho un montón de cosas muy malas.
>> She misses him, especially on rainy days.	Ella le echa de menos, especialmente en los días de lluvia.
>> What happened to Tom's head?	¿Qué le pasó a la cabeza de Tom?
>> A cat got out from under the car.	Un gato salió de abajo del auto.


In [7]:
# Q. 전처리 함수를 만들어 보세요. 아래 기능을 추가해주세요.
def preprocess_sentence(sentence):
    sentence = sentence.lower() # 대문자를 소문자로 변환
    sentence = re.sub(r' {2,}', ' ', sentence) # 둘 이상의 공백을 하나의 공백으로 치환
    sentence = sentence.strip() # 문자열 양 끝 공백 제거
    return sentence

In [8]:
spa_eng_sentences = list(map(preprocess_sentence, spa_eng_sentences))

print('슝=3')

슝=3


In [9]:
test_sentence_count = total_sentence_count // 200
print("Test Size: ", test_sentence_count)
print("\n")

train_spa_eng_sentences = spa_eng_sentences[:-test_sentence_count]
test_spa_eng_sentences = spa_eng_sentences[-test_sentence_count:]
print("Train Example:", len(train_spa_eng_sentences))
for sen in train_spa_eng_sentences[0:100][::20]:
    print(">>", sen)
print("\n")
print("Test Example:", len(test_spa_eng_sentences))
for sen in test_spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Test Size:  594


Train Example: 118370
>> tom held up his hands.	tom levantó sus manos.
>> i've done a lot of very bad things.	he hecho un montón de cosas muy malas.
>> she misses him, especially on rainy days.	ella le echa de menos, especialmente en los días de lluvia.
>> what happened to tom's head?	¿qué le pasó a la cabeza de tom?
>> a cat got out from under the car.	un gato salió de abajo del auto.


Test Example: 594
>> that doesn't mean that i'll stop doing it.	no significa que dejaré de hacerlo.
>> he listened to the music with his eyes closed.	con los ojos cerrados escuchó la música.
>> i congratulated him on passing the entrance exam.	lo felicité por pasar el examen de ingreso.
>> i'm not sure this is a good idea.	no estoy segura de que esto sea una buena idea.
>> i'm not convinced.	no estoy convencida.


In [10]:
def split_spa_eng_sentences(spa_eng_sentences):
    spa_sentences = []
    eng_sentences = []
    for spa_eng_sentence in tqdm(spa_eng_sentences):
        eng_sentence, spa_sentence = spa_eng_sentence.split('\t')
        spa_sentences.append(spa_sentence)
        eng_sentences.append(eng_sentence)
    return eng_sentences, spa_sentences

print('슝=3')

슝=3


In [11]:
train_eng_sentences, train_spa_sentences = split_spa_eng_sentences(train_spa_eng_sentences)
print(len(train_eng_sentences))
print(train_eng_sentences[0])
print('\n')
print(len(train_spa_sentences))
print(train_spa_sentences[0])

  0%|          | 0/118370 [00:00<?, ?it/s]

118370
tom held up his hands.


118370
tom levantó sus manos.


In [12]:
test_eng_sentences, test_spa_sentences = split_spa_eng_sentences(test_spa_eng_sentences)
print(len(test_eng_sentences))
print(test_eng_sentences[0])
print('\n')
print(len(test_spa_sentences))
print(test_spa_sentences[0])

  0%|          | 0/594 [00:00<?, ?it/s]

594
that doesn't mean that i'll stop doing it.


594
no significa que dejaré de hacerlo.


## 토큰화

In [13]:
def generate_tokenizer(corpus,
                       vocab_size,
                       lang="spa-eng",
                       pad_id=0,   # pad token의 일련번호
                       bos_id=1,  # 문장의 시작을 의미하는 bos token(<s>)의 일련번호
                       eos_id=2,  # 문장의 끝을 의미하는 eos token(</s>)의 일련번호
                       unk_id=3):   # unk token의 일련번호
    file = "./%s_corpus.txt" % lang
    model = "%s_spm" % lang

    with open(file, 'w') as f:
        for row in corpus: f.write(str(row) + '\n')

    import sentencepiece as spm
    spm.SentencePieceTrainer.Train(
        '--input=./%s --model_prefix=%s --vocab_size=%d'\
        % (file, model, vocab_size) + \
        '--pad_id=%d --bos_id=%d --eos_id=%d --unk_id=%d'\
        % (pad_id, bos_id, eos_id, unk_id)
    )

    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load('%s.model' % model)

    return tokenizer

print("슝=3")

슝=3


In [14]:
VOCAB_SIZE = 20000
tokenizer = generate_tokenizer(train_eng_sentences + train_spa_sentences, VOCAB_SIZE, 'spa-eng')
tokenizer.set_encode_extra_options("bos:eos")  # 문장 양 끝에 <s> , </s> 추가

True

In [15]:
def make_corpus(sentences, tokenizer):
    corpus = []
    for sentence in tqdm(sentences):
        tokens = tokenizer.encode_as_ids(sentence)
        corpus.append(tokens)
    return corpus

print('슝=3')

슝=3


In [16]:
eng_corpus = make_corpus(train_eng_sentences, tokenizer)
spa_corpus = make_corpus(train_spa_sentences, tokenizer)

  0%|          | 0/118370 [00:00<?, ?it/s]

  0%|          | 0/118370 [00:00<?, ?it/s]

In [17]:
print(train_eng_sentences[0])
print(eng_corpus[0])
print('\n')
print(train_spa_sentences[0])
print(spa_corpus[0])

tom held up his hands.
[1, 5, 2452, 106, 59, 1114, 0, 2]


tom levantó sus manos.
[1, 5, 2300, 210, 1143, 0, 2]


In [18]:
MAX_LEN = 50

def pad_sequences_custom(sequences, max_len=50, pad_value=0):
    """
    sequences: list of list (각 문장별 토큰 ID 리스트)
    max_len: 고정할 최대 시퀀스 길이
    pad_value: 패딩에 사용할 값 (일반적으로 0)
    """
    padded_sequences = []

    for seq in sequences:
        # 초과 길이는 자르고
        if len(seq) > max_len:
            seq = seq[:max_len]
        # 부족한 길이는 pad_value로 채우기
        else:
            seq = seq + [pad_value] * (max_len - len(seq))

        padded_sequences.append(seq)

    # 최종적으로 torch.Tensor로 변환 (shape: [batch_size, max_len])
    return torch.tensor(padded_sequences, dtype=torch.long)

enc_ndarray = pad_sequences_custom(eng_corpus, max_len=MAX_LEN, pad_value=0)
dec_ndarray = pad_sequences_custom(spa_corpus, max_len=MAX_LEN, pad_value=0)

print(enc_ndarray.shape)  # 예) [batch_size, 50]
print(dec_ndarray.shape)  # 예) [batch_size, 50]
print("슝=3")

torch.Size([118370, 50])
torch.Size([118370, 50])
슝=3


In [19]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = TensorDataset(enc_ndarray, dec_ndarray)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

print("슝=3")

슝=3


# 3. 번역모델 만들기

## 트랜스포머 구현하기

### Positional Encoding

In [20]:
# Positional Encoding 구현
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2*(i//2)) / np.float32(d_model))

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table
print("슝=3")

슝=3


### 마스크 생성

In [21]:
def generate_padding_mask(seq: torch.Tensor) -> torch.Tensor:
    """
    seq: shape [batch_size, seq_len]의 입력 (토큰 ID 텐서)
    반환: shape [batch_size, 1, 1, seq_len]의 패딩 마스크
         (seq == 0)인 위치가 1, 나머지는 0
    """
    # (seq == 0)은 불리언 텐서를 반환 -> float()로 형변환 -> (1.0 or 0.0)
    # 차원 확장: [batch_size, seq_len] → [batch_size, 1, 1, seq_len]
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size: int) -> torch.Tensor:
    """
    size: 문장(시퀀스) 길이
    반환: shape [size, size],
         i < j (대각선 위)에 해당하는 위치가 1, 아닌 곳은 0
         (미래 토큰을 가리기 위한 마스크)
    """
    # triu(diagonal=1)은 주대각선 위가 1, 아래가 0인 텐서를 만들어 줌
    return torch.triu(torch.ones(size, size), diagonal=1)


def generate_masks(src: torch.Tensor, tgt: torch.Tensor):
    """
    src, tgt: shape [batch_size, seq_len]
    3가지 마스크를 반환:
      - enc_mask: 인코더 입력용 패딩 마스크
      - dec_enc_mask: 디코더-인코더 어텐션용 패딩 마스크
      - dec_mask: 디코더 자기어텐션용 마스크(룩어헤드 + 패딩)

    각각의 shape:
      - enc_mask, dec_enc_mask: [batch_size, 1, 1, src_seq_len]
      - dec_mask: [batch_size, 1, tgt_seq_len, tgt_seq_len]
    """
    # 1) 인코더 입력용 패딩 마스크
    enc_mask = generate_padding_mask(src)
    # 2) 디코더에서 인코더 값을 볼 때 사용하는 마스크 (src 마스크 재사용)
    dec_enc_mask = generate_padding_mask(src)

    # 3) 디코더 자기어텐션 마스크 (미래 토큰 방지 룩어헤드 + tgt 자체 패딩 마스크)
    dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])  # [tgt_seq_len, tgt_seq_len]
    dec_tgt_padding_mask = generate_padding_mask(tgt)           # [batch_size, 1, 1, tgt_seq_len]

    # 룩어헤드 마스크를 (batch 차원과 head 차원을 가상으로) 확장
    dec_lookahead_mask = dec_lookahead_mask.unsqueeze(0).unsqueeze(1)  # [1, 1, seq_len, seq_len]

    # 패딩 + 룩어헤드 마스크 병합
    # 브로드캐스팅에 의해 shape [batch_size, 1, tgt_seq_len, tgt_seq_len]이 됨

    dec_tgt_padding_mask = dec_tgt_padding_mask.to(device)
    dec_lookahead_mask = dec_lookahead_mask.to(device)

    dec_mask = torch.max(dec_tgt_padding_mask, dec_lookahead_mask)

    return enc_mask, dec_enc_mask, dec_mask

print("슝=3")

슝=3


### Multi-Head Attention

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model을 num_heads로 나눈 만큼이 각 head가 담당할 차원 수
        self.depth = d_model // num_heads

        # Query, Key, Value를 구하는 선형 레이어
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 최종적으로 head들의 출력을 결합해주는 선형 레이어
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q, K, V:  [batch_size, num_heads, seq_len, depth]
        mask:     [batch_size, 1, seq_len, seq_len] 혹은
                  [batch_size, num_heads, seq_len, seq_len]
                  (어텐션에서 제외할 위치=1, 사용할 위치=0)
        """
        # d_k = depth
        d_k = Q.size(-1)  # K.shape[-1]도 동일
        # Q와 K의 전치 곱: (batch_size, num_heads, seq_len, seq_len)
        QK = torch.matmul(Q, K.transpose(-1, -2))

        # 스케일링
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 마스크가 있는 경우 -1e9(매우 작은 수)를 더하여 softmax 후 확률이 0에 가깝도록 처리
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)

        attentions = F.softmax(scaled_qk, dim=-1)  # (batch_size, num_heads, seq_len, seq_len)
        out = torch.matmul(attentions, V)         # (batch_size, num_heads, seq_len, depth)

        return out, attentions

    def split_heads(self, x):
        """
        x: [batch_size, seq_len, d_model]
        반환: [batch_size, num_heads, seq_len, depth]
        """
        bsz, seq_len, _ = x.size()
        # d_model -> (num_heads * depth)이므로 view로 재배치
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        # (batch_size, seq_len, num_heads, depth) -> (batch_size, num_heads, seq_len, depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        """
        x: [batch_size, num_heads, seq_len, depth]
        반환: [batch_size, seq_len, d_model]
        """
        bsz, num_heads, seq_len, depth = x.size()
        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch_size, seq_len, d_model]
        mask:    [batch_size, 1, seq_len, seq_len] 혹은
                 [batch_size, num_heads, seq_len, seq_len]
        """
        # W_q, W_k, W_v는 각각 (d_model -> d_model) 선형 변환
        WQ = self.W_q(Q)  # [batch_size, seq_len, d_model]
        WK = self.W_k(K)  # [batch_size, seq_len, d_model]
        WV = self.W_v(V)  # [batch_size, seq_len, d_model]

        # 멀티헤드 분할
        WQ_splits = self.split_heads(WQ)  # [batch_size, num_heads, seq_len, depth]
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # Scaled dot-product attention
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask
        )

        # head 결과 결합 후 최종 선형
        out = self.combine_heads(out)  # [batch_size, seq_len, d_model]
        out = self.linear(out)         # [batch_size, seq_len, d_model]

        return out, attention_weights

print("슝=3")

슝=3


### Position-wise Feed Forward Network

In [23]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.fc1(x))  # 첫 번째 Dense + ReLU
        out = self.fc2(out)          # 두 번째 Dense
        return out

print("슝=3")

슝=3


### Encoder Layer

In [24]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        # nn.LayerNorm은 마지막 차원(d_model)을 기준으로 정규화
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Multi-Head Attention 단계
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.do(out)
        out = out + residual  # residual connection

        # Position-Wise Feed Forward 단계
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual  # residual connection

        return out, enc_attn

print("슝=3")

슝=3


### Decoder Layer

In [25]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        # Masked Multi-Head Attention
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, mask=padding_mask)
        out = self.do(out)
        out = out + residual

        # Encoder-Decoder Multi-Head Attention (주의: Q, K, V 순서)
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out)
        out = out + residual

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual

        return out, dec_attn, dec_enc_attn

print("슝=3")

슝=3


### Encoder

In [26]:
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.do = nn.Dropout(dropout)  # 필요 시 입력에 dropout 적용 가능

    def forward(self, x, mask):
        out = x
        enc_attns = []
        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)
        return out, enc_attns

# 사용 예시: Encoder 인스턴스 생성 후 forward 호출
# encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
# out, enc_attns = encoder(x, mask)
print("슝=3")

슝=3


### Decoder

In [27]:
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        out = x
        dec_attns = []
        dec_enc_attns = []
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](out, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)
        return out, dec_attns, dec_enc_attns

print("슝=3")

슝=3


### Transformer 전체 모델 조립

In [28]:
import math

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared_fc=True, shared_emb=False):
        super(Transformer, self).__init__()
        # d_model은 스케일링에 사용되므로 float으로 저장
        self.d_model = float(d_model)

        # Embedding 레이어: shared_emb True면 동일한 임베딩을 사용합니다.
        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding (넘파이 버전 결과를 torch.Tensor로 변환)
        pos_encoding_np = positional_encoding(pos_len, d_model)
        # 파라미터로 등록하지 않고 고정값이므로 buffer로 등록합니다.
        self.register_buffer("pos_encoding", torch.tensor(pos_encoding_np, dtype=torch.float32))

        self.do = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared_fc = shared_fc
        if shared_fc:
            # fc 레이어와 디코더 임베딩의 weight를 공유합니다.
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        """
        emb: 임베딩 레이어
        x: [batch_size, seq_len] (토큰 인덱스)
        """
        seq_len = x.size(1)
        out = emb(x)  # [batch_size, seq_len, d_model]
        if self.shared_fc:
            out = out * math.sqrt(self.d_model)
        # pos_encoding: [pos_len, d_model] → [1, pos_len, d_model] 후 슬라이싱
        out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = self.do(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        """
        enc_in: [batch_size, src_seq_len]
        dec_in: [batch_size, tgt_seq_len]
        enc_mask, dec_enc_mask, dec_mask: 마스킹 텐서들
        """
        # Embedding 및 positional encoding 적용
        enc_in_emb = self.embedding(self.enc_emb, enc_in)
        dec_in_emb = self.embedding(self.dec_emb, dec_in)

        # Encoder와 Decoder 통과
        enc_out, enc_attns = self.encoder(enc_in_emb, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in_emb, enc_out, dec_enc_mask, dec_mask)

        logits = self.fc(dec_out)
        return logits, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


### 모델 인스턴스 생성

In [29]:
# 주어진 하이퍼파라미터로 Transformer 인스턴스 생성
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200,
    dropout=0.3,
    shared_fc=True,
    shared_emb=True)

transformer = transformer.to(device)

d_model = 512

print("슝=3")

슝=3


### Learning Rate Scheduler

In [30]:
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=60): # 4000
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        # step을 float으로 변환하여 지수 연산이 제대로 수행되도록 함
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

print("슝=3")

슝=3


### Learning Rate & Optimizer

In [31]:
# Learning Rate 인스턴스 선언
learning_rate = LearningRateScheduler(d_model)

# 초기 lr은 스텝 1에 해당하는 값으로 설정합니다.
optimizer = torch.optim.Adam(transformer.parameters(),
                             lr=learning_rate(1),
                             betas=(0.9, 0.98),
                             eps=1e-9)

print("슝=3")

슝=3


### Loss Function 정의

In [32]:
def loss_function(real, pred):
    """
    real: [batch_size, seq_len] (정답 토큰 인덱스)
    pred: [batch_size, seq_len, num_classes] (모델의 raw logits)
    """

    real = real.to(device)
    pred = pred.to(device)

    # 예측 값을 (N, C) 형태로 flatten하고, 정답도 flatten하여 개별 손실 값을 구함
    loss_ = F.cross_entropy(pred.contiguous().view(-1, pred.size(-1)), real.contiguous().view(-1), reduction='none')
    # 다시 (batch_size, seq_len)로 reshape
    loss_ = loss_.view(real.size())

    # real이 0이 아닌 위치에 대한 마스크 생성 (0이면 패딩 토큰)
    mask = (real != 0).float()
    loss_ = loss_ * mask

    # 전체 손실 합을 마스크 합으로 나누어 평균 손실 계산
    return loss_.sum() / mask.sum()

print("슝=3")

슝=3


### Train Step 정의

In [33]:
def train_step(src, tgt, model, optimizer):
    model.train()  # 모델을 training 모드로 전환
    optimizer.zero_grad()

    # tgt의 오른쪽 시프트: decoder input과 gold target 분리
    tgt_in = tgt[:, :-1]  # Decoder의 입력
    gold = tgt[:, 1:]     # Decoder의 정답(target)

    # 마스크 생성 (generate_masks는 PyTorch용으로 변환된 함수여야 합니다)
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

    src = src.to(device)
    tgt_in = tgt_in.to(device)
    enc_mask = enc_mask.to(device)
    dec_enc_mask = dec_enc_mask.to(device)
    dec_mask = dec_mask.to(device)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)

    # loss 계산
    loss = loss_function(gold, predictions)

    # 역전파 수행 및 파라미터 업데이트
    loss.backward()
    optimizer.step()

    return loss, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


### 훈련

In [34]:
%%time

EPOCHS = 3

for epoch in range(EPOCHS):
    total_loss = 0.0
    dataset_count = len(train_dataloader)  # train_loader는 PyTorch DataLoader입니다.
    tqdm_bar = tqdm(total=dataset_count)

    for batch, (src, tgt) in enumerate(train_dataloader):
        # train_step 함수는 (loss, enc_attns, dec_attns, dec_enc_attns)를 반환합니다.
        loss, enc_attns, dec_attns, dec_enc_attns = train_step(src, tgt, transformer, optimizer)

        total_loss += loss.item()  # PyTorch에서는 loss.numpy() 대신 loss.item() 사용
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)

    tqdm_bar.close()
    print(f"Epoch {epoch+1}, Loss: {total_loss / dataset_count:.4f}")

  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 1, Loss: 3945.1806


  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 2, Loss: 2577.4474


  0%|          | 0/1850 [00:00<?, ?it/s]

Epoch 3, Loss: 2055.4922
CPU times: user 14min 36s, sys: 2.89 s, total: 14min 39s
Wall time: 14min 54s


# 4. 번역 성능 측정하기

## BLEU Score

### NLTK 활용한 BLEU Score

In [35]:
# 아래 두 문장을 바꿔가며 테스트 해보세요
reference = "많 은 자연어 처리 연구자 들 이 트랜스포머 를 선호 한다".split()
candidate = "적 은 자연어 학 개발자 들 가 트랜스포머 을 선호 한다 요".split()

print("원문:", reference)
print("번역문:", candidate)
print("BLEU Score:", sentence_bleu([reference], candidate))

원문: ['많', '은', '자연어', '처리', '연구자', '들', '이', '트랜스포머', '를', '선호', '한다']
번역문: ['적', '은', '자연어', '학', '개발자', '들', '가', '트랜스포머', '을', '선호', '한다', '요']
BLEU Score: 8.190757052088229e-155


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [36]:
print("1-gram:", sentence_bleu([reference], candidate, weights=[1, 0, 0, 0]))
print("2-gram:", sentence_bleu([reference], candidate, weights=[0, 1, 0, 0]))
print("3-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 1, 0]))
print("4-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 0, 1]))

1-gram: 0.5
2-gram: 0.18181818181818182
3-gram: 2.2250738585072626e-308
4-gram: 2.2250738585072626e-308


In [37]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                         candidate,
                         weights=weights,
                         smoothing_function=SmoothingFunction().method1)  # smoothing_function 적용

print("BLEU-1:", calculate_bleu(reference, candidate, weights=[1, 0, 0, 0]))
print("BLEU-2:", calculate_bleu(reference, candidate, weights=[0, 1, 0, 0]))
print("BLEU-3:", calculate_bleu(reference, candidate, weights=[0, 0, 1, 0]))
print("BLEU-4:", calculate_bleu(reference, candidate, weights=[0, 0, 0, 1]))

print("\nBLEU-Total:", calculate_bleu(reference, candidate))

BLEU-1: 0.5
BLEU-2: 0.18181818181818182
BLEU-3: 0.010000000000000004
BLEU-4: 0.011111111111111112

BLEU-Total: 0.05637560315259291


In [38]:
import torch
import torch.nn.functional as F

def translate(tokens, model, src_tokenizer, tgt_tokenizer):
    # tokens: 입력 토큰 리스트
    # MAX_LEN: 최대 길이 (전역 변수 혹은 상수)
    # device: 모델과 데이터가 위치한 디바이스

    # tokens 길이가 MAX_LEN보다 크면 자르고, 작으면 0으로 패딩
    if len(tokens) > MAX_LEN:
        tokens = tokens[:MAX_LEN]
    else:
        tokens = tokens + [0] * (MAX_LEN - len(tokens))

    # 배치 차원을 추가하여 텐서로 변환 (shape: [1, MAX_LEN])
    padded_tokens = torch.tensor([tokens], dtype=torch.long, device=device)

    ids = []
    # 디코더의 첫 입력은 BOS 토큰 (배치 차원 추가)
    output = torch.tensor([[tgt_tokenizer.bos_id()]], dtype=torch.long, device=device)

    for i in range(MAX_LEN):
        # generate_masks는 padded_tokens와 현재 output으로부터 마스크들을 생성합니다.
        enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(padded_tokens, output)

        # 모델 예측: predictions shape: [batch, seq_len, num_classes]
        predictions, _, _, _ = model(padded_tokens, output, enc_padding_mask, combined_mask, dec_padding_mask)

        # 마지막 시퀀스 위치의 예측값을 소프트맥스 후 argmax로 선택
        predicted_id = predictions[0, -1].softmax(dim=-1).argmax(dim=-1).item()

        # EOS 토큰에 도달하면 현재까지의 예측 토큰 ids를 디코딩 후 반환
        if tgt_tokenizer.eos_id() == predicted_id:
            result = tgt_tokenizer.decode_ids(ids)
            return result

        ids.append(predicted_id)
        # 현재 output에 새로운 예측 토큰을 연결 (dim=1)
        new_token = torch.tensor([[predicted_id]], dtype=torch.long, device=device)
        output = torch.cat([output, new_token], dim=1)

    result = tgt_tokenizer.decode_ids(ids)
    return result

print("슝=3")

슝=3


In [39]:
def eval_bleu_single(model, src_sentence, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    src_tokens = src_tokenizer.encode_as_ids(src_sentence)
    tgt_tokens = tgt_tokenizer.encode_as_ids(tgt_sentence)

    if (len(src_tokens) > MAX_LEN): return None
    if (len(tgt_tokens) > MAX_LEN): return None

    reference = tgt_sentence.split()
    candidate = translate(src_tokens, model, src_tokenizer, tgt_tokenizer).split()

    score = sentence_bleu([reference], candidate,
                          smoothing_function=SmoothingFunction().method1)

    if verbose:
        print("Source Sentence: ", src_sentence)
        print("Model Prediction: ", candidate)
        print("Real: ", reference)
        print("Score: %lf\n" % score)

    return score

print('슝=3')

슝=3


In [40]:
# Q. 인덱스를 바꿔가며 테스트해 보세요
test_idx = 0

eval_bleu_single(transformer,
                 test_eng_sentences[test_idx],
                 test_spa_sentences[test_idx],
                 tokenizer,
                 tokenizer)

Source Sentence:  that doesn't mean that i'll stop doing it.
Model Prediction:  ['estoy', 'advertí', 'un', 'poco', 'de', 'tom', 'carbón', 'carbón', 'a', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious', 'delicious']
Real:  ['no', 'significa', 'que', 'dejaré', 'de', 'hacerlo.']
Score: 0.003668



0.00366753025392797

In [41]:
def eval_bleu(model, src_sentences, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    total_score = 0.0
    sample_size = len(src_sentences)

    for idx in tqdm(range(sample_size)):
        score = eval_bleu_single(model, src_sentences[idx], tgt_sentence[idx], src_tokenizer, tgt_tokenizer, verbose)
        if not score: continue

        total_score += score

    print("Num of Sample:", sample_size)
    print("Total Score:", total_score / sample_size)

print("슝=3")

슝=3


In [42]:
eval_bleu(transformer, test_eng_sentences, test_spa_sentences, tokenizer, tokenizer, verbose=False)

  0%|          | 0/594 [00:00<?, ?it/s]

Num of Sample: 594
Total Score: 0.003562679630520105


##  Beam Search Decoder

In [43]:
def beam_search_decoder(prob, beam_size):
    sequences = [[[], 1.0]]  # 생성된 문장과 점수를 저장

    for tok in prob:
        all_candidates = []

        for seq, score in sequences:
            for idx, p in enumerate(tok): # 각 단어의 확률을 총점에 누적 곱
                candidate = [seq + [idx], score * -math.log(-(p-1))]
                all_candidates.append(candidate)

        ordered = sorted(all_candidates,
                         key=lambda tup:tup[1],
                         reverse=True) # 총점 순 정렬
        sequences = ordered[:beam_size] # Beam Size에 해당하는 문장만 저장

    return sequences

print("슝=3")

슝=3


In [44]:
vocab = {
    0: "<pad>",
    1: "까요?",
    2: "커피",
    3: "마셔",
    4: "가져",
    5: "될",
    6: "를",
    7: "한",
    8: "잔",
    9: "도",
}

prob_seq = [[0.01, 0.01, 0.60, 0.32, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.75, 0.01, 0.01, 0.17],
            [0.01, 0.01, 0.01, 0.35, 0.48, 0.10, 0.01, 0.01, 0.01, 0.01],
            [0.24, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.68],
            [0.01, 0.01, 0.12, 0.01, 0.01, 0.80, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.81, 0.01, 0.01, 0.01, 0.01, 0.11, 0.01, 0.01, 0.01],
            [0.70, 0.22, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01]]

prob_seq = np.array(prob_seq)
beam_size = 3

result = beam_search_decoder(prob_seq, beam_size)

for seq, score in result:
    sentence = ""

    for word in seq:
        sentence += vocab[word] + " "

    print(sentence, "// Score: %.4f" % score)

커피 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 42.5243
커피 를 마셔 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 28.0135
마셔 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 17.8983


### Beam Search Decoder 작성 및 평가하기

In [45]:
import torch
import torch.nn.functional as F

def calc_prob(src_ids, tgt_ids, model):
    # 마스크 생성 (PyTorch 버전)
    enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(src_ids, tgt_ids)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(
        src_ids,
        tgt_ids,
        enc_padding_mask,
        combined_mask,
        dec_padding_mask
    )

    # 마지막 차원에 대해 softmax 적용하여 확률값 계산
    return F.softmax(predictions, dim=-1)

print("슝=3")

슝=3


In [46]:
import numpy as np
import torch

def beam_search_decoder(sentence,
                        src_len,
                        tgt_len,
                        model,
                        src_tokenizer,
                        tgt_tokenizer,
                        beam_size):
    # 입력 문장을 토큰화
    tokens = src_tokenizer.encode_as_ids(sentence)

    # src_in: [1, src_len] 크기의 텐서로 padding (0: 패딩 토큰)
    padded = np.zeros((1, src_len), dtype=np.int64)
    padded[0, :len(tokens)] = tokens
    src_in = torch.tensor(padded, dtype=torch.long, device=device)

    # beam search용 캐시 배열들
    pred_cache = np.zeros((beam_size * beam_size, tgt_len), dtype=np.int64)
    pred_tmp = np.zeros((beam_size, tgt_len), dtype=np.int64)

    eos_flag = np.zeros((beam_size,), dtype=np.int64)  # EOS를 만난 branch 표시 (EOS: -1)
    scores = np.ones((beam_size,), dtype=np.float32)     # 각 branch의 score (확률 곱)

    # 디코더 첫 입력은 BOS 토큰
    pred_tmp[:, 0] = tgt_tokenizer.bos_id()

    # 초기 디코더 입력 (branch 0의 첫 토큰) -> shape: [1, 1]
    dec_in = torch.tensor(pred_tmp[0, :1], dtype=torch.long, device=device).unsqueeze(0)
    # calc_prob()는 softmax를 적용한 확률 텐서를 반환함
    prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

    # seq_pos: 디코더 시퀀스 위치
    for seq_pos in range(1, tgt_len):
        score_cache = np.ones((beam_size * beam_size,), dtype=np.float32)

        # 각 beam branch에 대해 캐시 초기화
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            score_cache[cache_pos:cache_pos+beam_size] = scores[branch_idx]
            pred_cache[cache_pos:cache_pos+beam_size, :seq_pos] = pred_tmp[branch_idx, :seq_pos]

        # 각 beam branch에 대해 후보 확률 계산 및 캐시 업데이트
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            if seq_pos != 1:
                # 해당 branch의 현재까지의 시퀀스를 디코더 입력으로 변환
                dec_in_np = pred_cache[branch_idx, :seq_pos]
                dec_in = torch.tensor(dec_in_np, dtype=torch.long, device=device).unsqueeze(0)
                prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

            # 각 branch 내에서 beam_size만큼의 후보 토큰을 선택
            for beam_idx in range(beam_size):
                max_idx = np.argmax(prob)
                # 후보 branch의 score 업데이트 (곱셈으로 누적)
                score_cache[cache_pos + beam_idx] *= prob[max_idx]
                pred_cache[cache_pos + beam_idx, seq_pos] = max_idx
                # 이미 선택된 토큰은 다시 선택되지 않도록 -1로 마킹
                prob[max_idx] = -1

        # 각 beam branch에서 최고 score를 가진 후보를 선택
        for beam_idx in range(beam_size):
            if eos_flag[beam_idx] == -1:
                continue
            max_idx = np.argmax(score_cache)
            prediction = pred_cache[max_idx, :seq_pos+1].copy()
            pred_tmp[beam_idx, :seq_pos+1] = prediction
            scores[beam_idx] = score_cache[max_idx]
            score_cache[max_idx] = -1  # 해당 후보 제거

            # 만약 EOS 토큰이면 해당 branch는 종료 표시 (-1)
            if prediction[-1] == tgt_tokenizer.eos_id():
                eos_flag[beam_idx] = -1

    # 각 branch의 예측 시퀀스에서 EOS 토큰 이전까지만 추출하여 결과 반환
    pred = []
    for long_pred in pred_tmp:
        eos_token = tgt_tokenizer.eos_id()
        # EOS 토큰이 없는 경우, 전체 시퀀스를 사용하도록 처리할 수 있음
        try:
            eos_idx = list(long_pred).index(eos_token)
        except ValueError:
            eos_idx = tgt_len - 1
        short_pred = long_pred[:eos_idx+1]
        pred.append(short_pred.tolist())

    return pred

print("슝=3")

슝=3


In [47]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                            candidate,
                            weights=weights,
                            smoothing_function=SmoothingFunction().method1)

print('슝=3')

슝=3


In [48]:
def beam_bleu(reference, ids, tokenizer):
    # 기준 문장을 토큰화
    reference_tokens = reference.split()

    total_score = 0.0
    num_candidates = len(ids)
    if num_candidates == 0:
        return 0.0

    for candidate_ids in ids:
        # 후보 문장을 디코딩 후 토큰화
        candidate_sentence = tokenizer.decode_ids(candidate_ids)
        candidate_tokens = candidate_sentence.split()

        score = calculate_bleu(reference_tokens, candidate_tokens)

        print(f"Reference: {reference_tokens}")
        print(f"Candidate: {candidate_tokens}")
        print(f"BLEU: {score}")

        total_score += score

    return total_score / num_candidates

print("슝=3")

슝=3


In [49]:
# Q. 인덱스를 바꿔가며 확인해 보세요
test_idx = 1

ids = \
beam_search_decoder(test_eng_sentences[test_idx],
                    MAX_LEN,
                    MAX_LEN,
                    transformer,
                    tokenizer,
                    tokenizer,
                    beam_size=5)

bleu = beam_bleu(test_spa_sentences[test_idx], ids, tokenizer)
print(bleu)

Reference: ['soy', 'tu', 'vecino.']
Candidate: ['estoy', 'artificial', 'artificial', 'de', 'estoy', 'dilo', 'dilo', 'dilo', 'dilo', 'dilo', 'dilo', 'dilo', 'dilo', 'que', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes']
BLEU: 0
Reference: ['soy', 'tu', 'vecino.']
Candidate: ['estoy', 'artificial', 'artificial', 'de', 'mire', 'electronic', 'dilo', 'dilo', 'de', 'dilo', 'que', 'dilo', 'a,', 'dispute', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes', 'lastimes'

# 5. 데이터 부풀리기

In [50]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 51.2 MB/s  0:00:00


In [51]:
import logging
import gensim.downloader as api

# 로그 출력을 최소화 (오류만 표시)
logging.getLogger('gensim').setLevel(logging.WARNING)

# 모델 로드
wv = api.load('glove-wiki-gigaword-300')

[==================================================] 100.0% 376.1/376.1MB downloaded


In [52]:
wv.most_similar("banana")

[('bananas', 0.6691170930862427),
 ('mango', 0.5804104208946228),
 ('pineapple', 0.5492372512817383),
 ('coconut', 0.5462778806686401),
 ('papaya', 0.541056752204895),
 ('fruit', 0.52181077003479),
 ('growers', 0.4877638816833496),
 ('nut', 0.48399588465690613),
 ('peanut', 0.48062023520469666),
 ('potato', 0.48061180114746094)]

In [53]:
sample_sentence = "you know ? all you need is attention ."
sample_tokens = sample_sentence.split()

selected_tok = random.choice(sample_tokens)

result = ""
for tok in sample_tokens:
    if tok is selected_tok:
        result += wv.most_similar(tok)[0][0] + " "

    else:
        result += tok + " "

print("From:", sample_sentence)
print("To:", result)

From: you know ? all you need is attention .
To: you know ? all you need is focus . 


### Lexical Substitution 구현하기

In [54]:
# Q. Lexical Substitution 을 구현해봅시다.
def lexical_sub(sentence, wv):
    # 문장을 토큰화
    tokens = sentence.split()

    # 유효한 단어 필터링 (임베딩에 존재하는 단어만 고려)
    valid_tokens = [tok for tok in tokens if tok in wv]

    # 대체할 단어 선택 (임베딩 내 존재하는 단어 중 하나)
    if not valid_tokens:
        return sentence  # 모든 단어가 임베딩 내에 없으면 원래 문장 반환

    selected_tok = random.choice(valid_tokens)

    # 가장 유사한 단어 찾기
    similar_word = wv.most_similar(selected_tok)[0][0]

    # 변환된 문장 생성
    new_sentence = " ".join([similar_word if tok == selected_tok else tok for tok in tokens])

    return new_sentence

In [55]:
new_corpus = []

for old_src in tqdm(test_eng_sentences):
    new_src = lexical_sub(old_src, wv)
    if new_src is not None:
        new_corpus.append(new_src)
    # Augmentation이 없더라도 원본 문장을 포함시킵니다
    new_corpus.append(old_src)

print(new_corpus[:10])

  0%|          | 0/594 [00:00<?, ?it/s]

["that doesn't mean that i'll stop done it.", "that doesn't mean that i'll stop doing it.", "i'm my neighbor.", "i'm your neighbor.", 'tom did another excellent job.', 'tom did an excellent job.', "this box won't fit where my suitcase.", "this box won't fit in my suitcase.", "do 'll have beer?", 'do you have beer?']


# Project: 멋진 챗봇 만들기

In [56]:
import numpy
import pandas
import torch
import nltk
import gensim

print(numpy.__version__)
print(pandas.__version__)
print(torch.__version__)
print(nltk.__version__)
print(gensim.__version__)

2.0.2
2.2.2
2.10.0+cu128
3.9.1
4.4.0


## Step 1. 데이터 다운로드

In [57]:
# 필요 라이브러리 임포트
import numpy as np
import pandas as pd
import torch
import nltk
import gensim
import re
import os
import random
from konlpy.tag import Mecab
from collections import Counter
from tqdm import tqdm
import urllib.request
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import math
from nltk.translate.bleu_score import sentence_bleu

# Step 1. 데이터 다운로드 및 불러오기
# 데이터가 없을 경우 로컬로 다운로드합니다.
if not os.path.exists('ChatbotData.csv'):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv", filename="ChatbotData.csv")

# pandas를 사용하여 csv 파일 읽기
train_data = pd.read_csv('ChatbotData.csv')

# 질문과 답변을 각각 questions, answers 변수에 저장
questions = train_data['Q'].values
answers = train_data['A'].values

## Step 2. 데이터 정제

In [58]:
# Step 2. 데이터 정제
def preprocess_sentence(sentence):
    # 1. 영문자의 경우 모두 소문자로 변환
    sentence = sentence.lower()
    # 2. 영문자, 한글, 숫자, 주요 특수문자(?, !, ., ,)를 제외하고 모두 제거하기 위해 정규식 사용
    sentence = re.sub(r'[^a-zA-Z가-힣0-9?!.,\s]', '', sentence)
    return sentence

## Step 3. 데이터 토큰화

In [60]:
# Step 3. 데이터 토큰화
mecab = Mecab()

def build_corpus(src_data, tgt_data, max_len=40):
    que_corpus = []
    ans_corpus = []

    # 중복 검사를 위해 set 자료구조 활용 (검사 속도 향상)
    que_set = set()
    ans_set = set()

    for src, tgt in zip(src_data, tgt_data):
        # 정제 함수 적용
        src = preprocess_sentence(src)
        tgt = preprocess_sentence(tgt)

        # mecab.morphs를 사용한 토큰화
        src_tokens = mecab.morphs(src)
        tgt_tokens = mecab.morphs(tgt)

        # 일정 길이(max_len) 이상인 문장 제외
        if len(src_tokens) < max_len and len(tgt_tokens) < max_len:
            # 리스트를 튜플로 변환하여 set에 추가 및 검색이 가능하도록 변경
            src_tuple = tuple(src_tokens)
            tgt_tuple = tuple(tgt_tokens)

            # 소스와 타겟 각각 중복이 아닐 경우에만 코퍼스에 추가
            if src_tuple not in que_set and tgt_tuple not in ans_set:
                que_set.add(src_tuple)
                ans_set.add(tgt_tuple)

                que_corpus.append(src_tokens)
                ans_corpus.append(tgt_tokens)

    return que_corpus, ans_corpus

que_corpus, ans_corpus = build_corpus(questions, answers)

## Step 4. Augmentation

In [70]:
# Step 4. Augmentation (Lexical Substitution)
# !pip install gensim==3.8.3
# word2vec = gensim.models.Word2Vec.load('ko.bin')

def lexical_sub(sentence, word2vec, threshold=0.5):
    # 원본 토큰 리스트 복사
    res = sentence.copy()
    try:
        # 문장에서 무작위로 교체할 단어의 인덱스 선택
        idx = random.randint(0, len(res) - 1)
        word = res[idx]
        # 해당 단어가 word2vec 모델 어휘 사전에 있는지 확인
        if word in word2vec.wv:
            # 가장 유사한 단어 추출
            similar_words = word2vec.wv.most_similar(word)
            for sim_word, score in similar_words:
                # 유사도가 임계값(threshold) 이상일 경우 치환하고 종료
                if score > threshold:
                    res[idx] = sim_word
                    break
    except Exception as e:
        pass
    return res

# Augmentation 데이터 생성 (Word2Vec 모델 로드 후 주석 해제하여 사용)
# final_que_corpus = que_corpus.copy()
# final_ans_corpus = ans_corpus.copy()

# for que, ans in tqdm(zip(que_corpus, ans_corpus), total=len(que_corpus)):
#     # 질문 Augmentation, 답변 원본 유지
#     aug_que = lexical_sub(que, word2vec)
#     final_que_corpus.append(aug_que)
#     final_ans_corpus.append(ans)
#
#     # 질문 원본 유지, 답변 Augmentation
#     aug_ans = lexical_sub(ans, word2vec)
#     final_que_corpus.append(que)
#     final_ans_corpus.append(aug_ans)

# (테스트 환경에서는 원본 데이터로 진행)
final_que_corpus = que_corpus
final_ans_corpus = ans_corpus

## Step 5. 데이터 벡터화

In [71]:
# Step 5. 데이터 벡터화
tgt_corpus = []
# 타겟 데이터(ans_corpus) 양 끝에 <start>와 <end> 토큰 추가
for ans in final_ans_corpus:
    tgt_corpus.append(["<start>"] + ans + ["<end>"])

# 소스와 타겟 데이터를 모두 포함하여 전체 단어 사전 구축
vocab_list = ["<pad>", "<unk>", "<start>", "<end>"]
words = []
for que in final_que_corpus:
    words.extend(que)
for tgt in tgt_corpus:
    words.extend(tgt)

word_counts = Counter(words)

# 등장 빈도순으로 정렬하여 단어장 구축
for word, count in word_counts.most_common(10000):
    if word not in vocab_list:
        vocab_list.append(word)

word_to_index = {word: index for index, word in enumerate(vocab_list)}
index_to_word = {index: word for index, word in enumerate(vocab_list)}

def text_to_sequence(corpus, word_to_index):
    # 텍스트 코퍼스를 정수 시퀀스로 변환. 모르는 단어는 <unk> 처리.
    sequences = []
    for sentence in corpus:
        seq = [word_to_index.get(word, word_to_index["<unk>"]) for word in sentence]
        sequences.append(seq)
    return sequences

# 변환 수행
enc_train = text_to_sequence(final_que_corpus, word_to_index)
dec_train = text_to_sequence(tgt_corpus, word_to_index)

# 시퀀스 길이 패딩 (길이가 짧은 문장을 maxlen으로 맞추고 0으로 채움)
max_len = 40
def pad_sequences(sequences, maxlen, padding_value=0):
    padded = np.full((len(sequences), maxlen), padding_value, dtype=int)
    for i, seq in enumerate(sequences):
        length = min(len(seq), maxlen)
        padded[i, :length] = seq[:length]
    return padded

enc_train = pad_sequences(enc_train, maxlen=max_len, padding_value=word_to_index["<pad>"])
dec_train = pad_sequences(dec_train, maxlen=max_len, padding_value=word_to_index["<pad>"])

## Step 6. 훈련하기

In [73]:
# Step 6. 훈련하기
# Dataset 및 DataLoader 구성
class ChatbotDataset(Dataset):
    def __init__(self, enc_data, dec_data):
        self.enc_data = torch.tensor(enc_data, dtype=torch.long)
        self.dec_data = torch.tensor(dec_data, dtype=torch.long)

    def __len__(self):
        return len(self.enc_data)

    def __getitem__(self, idx):
        return self.enc_data[idx], self.dec_data[idx]

# 하이퍼파라미터 세팅
BATCH_SIZE = 64
N_LAYERS = 1
D_MODEL = 368
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.2
EPOCHS = 10
VOCAB_SIZE = len(vocab_list)

dataset = ChatbotDataset(enc_train, dec_train)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Positional Encoding 정의
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

# Transformer 모델 정의
class TransformerModel(nn.Module):
    def __init__(self, ntoken, d_model, nhead, d_hid, nlayers, dropout=0.5):
        super(TransformerModel, self).__init__()
        self.pos_encoder = PositionalEncoding(d_model)
        self.embedding = nn.Embedding(ntoken, d_model)

        # 인코더와 디코더 레이어 구성
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, d_hid, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, nlayers)

        decoder_layers = nn.TransformerDecoderLayer(d_model, nhead, d_hid, dropout, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layers, nlayers)

        self.out = nn.Linear(d_model, ntoken)
        self.d_model = d_model

    def generate_square_subsequent_mask(self, sz):
        # 디코더의 Masking
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src, tgt, tgt_mask=None, src_padding_mask=None, tgt_padding_mask=None):
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)

        tgt = self.embedding(tgt) * math.sqrt(self.d_model)
        tgt = self.pos_encoder(tgt)

        memory = self.transformer_encoder(src, src_key_padding_mask=src_padding_mask)
        output = self.transformer_decoder(tgt, memory, tgt_mask=tgt_mask,
                                          tgt_key_padding_mask=tgt_padding_mask, memory_key_padding_mask=src_padding_mask)
        return self.out(output)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델, 손실 함수, 옵티마이저 초기화
model = TransformerModel(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word_to_index["<pad>"])
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [74]:
# 학습 과정 루프 (주석 해제 후 훈련 진행)
for epoch in range(EPOCHS):
     model.train()
     total_loss = 0
     for src, tgt in dataloader:
         src = src.to(device)
         tgt_input = tgt[:, :-1].to(device)
         tgt_real = tgt[:, 1:].to(device)

         tgt_mask = model.generate_square_subsequent_mask(tgt_input.size(1)).to(device)
         src_padding_mask = (src == word_to_index["<pad>"]).to(device)
         tgt_padding_mask = (tgt_input == word_to_index["<pad>"]).to(device)

         optimizer.zero_grad()
         output = model(src, tgt_input, tgt_mask=tgt_mask, src_padding_mask=src_padding_mask, tgt_padding_mask=tgt_padding_mask)

         loss = criterion(output.view(-1, VOCAB_SIZE), tgt_real.contiguous().view(-1))
         loss.backward()
         optimizer.step()
         total_loss += loss.item()
     print(f"Epoch: {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

def evaluate(sentence, model, max_len=40):
    model.eval()
    # 입력 문장 정제 및 토큰화
    sentence = preprocess_sentence(sentence)
    tokens = mecab.morphs(sentence)
    seq = [word_to_index.get(word, word_to_index["<unk>"]) for word in tokens]

    src = torch.tensor([seq]).to(device)
    tgt = torch.tensor([[word_to_index["<start>"]]]).to(device)

    # 디코딩 루프
    for i in range(max_len):
        tgt_mask = model.generate_square_subsequent_mask(tgt.size(1)).to(device)
        src_padding_mask = (src == word_to_index["<pad>"]).to(device)

        with torch.no_grad():
            output = model(src, tgt, tgt_mask=tgt_mask, src_padding_mask=src_padding_mask)

        prob = output[:, -1, :]
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        # <end> 토큰 예측 시 종료
        if next_word == word_to_index["<end>"]:
            break

        tgt = torch.cat([tgt, torch.tensor([[next_word]]).to(device)], dim=1)

    decoded_words = [index_to_word[idx.item()] for idx in tgt[0]][1:]
    return ' '.join(decoded_words)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch: 1, Loss: 5.9531
Epoch: 2, Loss: 4.5628
Epoch: 3, Loss: 4.1989
Epoch: 4, Loss: 3.9707
Epoch: 5, Loss: 3.8063
Epoch: 6, Loss: 3.6786
Epoch: 7, Loss: 3.5611
Epoch: 8, Loss: 3.4605
Epoch: 9, Loss: 3.3612
Epoch: 10, Loss: 3.2659


In [75]:
#예문 테스트 (모델 훈련 후 출력 확인)
examples = [
     "지루하다, 놀러가고 싶어.",
     "오늘 일찍 일어났더니 피곤하다.",
     "간만에 여자친구랑 데이트 하기로 했어.",
     "집에 있는다는 소리야."
 ]
for ex in examples:
     print(f"Q: {ex}\nA: {evaluate(ex, model)} <end>\n")

Q: 지루하다, 놀러가고 싶어.
A: 저 도 좋 은 사람 이 에요 . <end>

Q: 오늘 일찍 일어났더니 피곤하다.
A: 좋 은 사람 이 에요 . <end>

Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 좋 은 사람 이 에요 . <end>

Q: 집에 있는다는 소리야.
A: 마음 이 에요 . <end>



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


## Step 7. 성능 측정하기

In [76]:
# Step 7. 성능 측정하기
def calculate_bleu(reference, candidate, weights=(0.25, 0.25, 0.25, 0.25)):
    # reference는 리스트의 리스트 형태로 입력 [[ref_token1, ref_token2, ...]]
    # candidate는 리스트 형태로 입력 [cand_token1, cand_token2, ...]
    return sentence_bleu([reference], candidate, weights=weights)

In [77]:
# BLEU 계산
ref_tokens = mecab.morphs(preprocess_sentence("잠깐 쉬어도 돼요."))
cand_tokens = evaluate("지루하다, 놀러가고 싶어.", model).split()
bleu_score = calculate_bleu(ref_tokens, cand_tokens)
print(f"BLEU Score: {bleu_score}")

BLEU Score: 1.0832677820940877e-231


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

# ===========================================================

In [79]:
import torch
import torch.nn.functional as F

# [이전 결과의 문제점]
# 1. 학습 데이터의 절대적인 양이 부족함 (약 1만개).
# 2. 학습 에포크가 10으로 설정되어 있어 모델이 언어 패턴을 충분히 학습하지 못함.
# 3. Greedy Search(매 스텝 가장 높은 확률의 단어 하나만 선택) 방식으로 인해,
#    한 번 흔한 단어(예: '좋은', '사람')가 선택되면 이후 문맥이 단조로워지는 지역 최적해(Local Optima)에 빠짐.

# [수정 사항]
# 1. 디코딩 방식을 Greedy Search에서 Beam Search로 변경하여 확률이 높은 여러 후보(Beam)를 동시에 탐색하도록 수정.
# 2. evaluate_beam_search 함수를 새로 정의.
# 3. 모델 하이퍼파라미터 중 에폭 수를 늘려 추가 훈련을 진행할 수 있도록 변수 제공 (테스트 시 활용 가능).

In [80]:
# 하이퍼파라미터 수정
EPOCHS = 20

def evaluate_beam_search(sentence, model, beam_size=3, max_len=40):
    model.eval()

    # 1. 입력 문장 전처리 및 토큰화
    sentence = preprocess_sentence(sentence)
    tokens = mecab.morphs(sentence)
    seq = [word_to_index.get(word, word_to_index["<unk>"]) for word in tokens]

    src = torch.tensor([seq]).to(device)

    # 2. Beam Search 초기화
    # beams 리스트는 (누적 로그 확률, 시퀀스 리스트) 형태의 튜플을 저장
    start_token = word_to_index["<start>"]
    beams = [(0.0, [start_token])]

    for i in range(max_len):
        new_beams = []

        for score, seq_list in beams:
            # 이미 <end> 토큰에 도달한 빔은 그대로 유지
            if seq_list[-1] == word_to_index["<end>"]:
                new_beams.append((score, seq_list))
                continue

            # 현재까지의 시퀀스를 텐서로 변환
            tgt = torch.tensor([seq_list]).to(device)

            # 마스킹 생성
            tgt_mask = model.generate_square_subsequent_mask(tgt.size(1)).to(device)
            src_padding_mask = (src == word_to_index["<pad>"]).to(device)

            # 모델 예측
            with torch.no_grad():
                output = model(src, tgt, tgt_mask=tgt_mask, src_padding_mask=src_padding_mask)

            # 마지막 단어에 대한 확률 분포 (Log Softmax를 사용하여 덧셈 연산으로 확률 누적)
            prob = F.log_softmax(output[:, -1, :], dim=-1).squeeze(0)

            # 반복 방지 (Repetition Penalty)
            if len(seq_list) > 1:
                last_word = seq_list[-1]
                prob[last_word] -= 1e4

            # 확률이 가장 높은 상위 beam_size 개의 단어 추출
            topk_prob, topk_idx = torch.topk(prob, beam_size)

            # 새로운 빔 생성 및 누적 점수 계산
            for p, idx in zip(topk_prob, topk_idx):
                new_score = score + p.item()
                new_seq = seq_list + [idx.item()]
                new_beams.append((new_score, new_seq))

        # 3. 확장된 빔들 중 점수가 가장 높은 상위 beam_size 개만 남김
        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]

        # 4. 모든 빔이 <end> 토큰으로 끝났다면 탐색 종료
        if all(seq[-1] == word_to_index["<end>"] for _, seq in beams):
            break

    # 가장 높은 점수를 얻은 최종 빔 선택
    best_seq = beams[0][1]

    # <start> 및 <end> 토큰을 제외하고 문자열로 디코딩
    decoded_words = []
    for idx in best_seq:
        if idx == word_to_index["<start>"]:
            continue
        if idx == word_to_index["<end>"]:
            break
        decoded_words.append(index_to_word[idx])

    return ' '.join(decoded_words)

In [81]:
# 테스트 실행부
examples = [
     "지루하다, 놀러가고 싶어.",
     "오늘 일찍 일어났더니 피곤하다.",
     "간만에 여자친구랑 데이트 하기로 했어.",
     "집에 있는다는 소리야."
]
for ex in examples:
     print(f"Q: {ex}\nA: {evaluate_beam_search(ex, model, beam_size=3)} <end>\n")

Q: 지루하다, 놀러가고 싶어.
A: 저 도 모르 겠 어요 . <end>

Q: 오늘 일찍 일어났더니 피곤하다.
A: 잘 하 고 싶 어요 . <end>

Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 좋 은 사람 이 에요 . <end>

Q: 집에 있는다는 소리야.
A: 마음 이 에요 . <end>



In [83]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore")

class CustomSchedule(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps=1000, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super(CustomSchedule, self).__init__(optimizer, last_epoch)

    # 학습률을 계산하여 반환하는 함수
    def get_lr(self):
        step_num = self.last_epoch + 1
        arg1 = step_num ** -0.5
        arg2 = step_num * (self.warmup_steps ** -1.5)
        lr = (self.d_model ** -0.5) * min(arg1, arg2)
        return [lr for _ in self.base_lrs]

# 하이퍼파라미터 재설정
EPOCHS = 100
D_MODEL = 368
WARMUP_STEPS = 1000

# 옵티마이저 초기화 (이전 학습 상태를 덮어쓰고 새로 시작)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0, betas=(0.9, 0.98), eps=1e-9)

# 스케줄러 객체 생성
scheduler = CustomSchedule(optimizer, d_model=D_MODEL, warmup_steps=WARMUP_STEPS)

# 손실 함수 설정 (패딩 토큰은 손실값 계산에서 무시)
criterion = nn.CrossEntropyLoss(ignore_index=word_to_index["<pad>"])

print(f"모델을 {EPOCHS} Epoch 만큼 학습")

# 100 Epoch 학습 루프 시작
for epoch in range(EPOCHS):
    model.train() # 모델을 학습 모드로 변경
    total_loss = 0

    # 데이터로더를 통해 배치 단위로 학습 데이터 로드
    for src, tgt in dataloader:
        src = src.to(device)
        tgt_input = tgt[:, :-1].to(device) # 마지막 <end> 토큰 제외
        tgt_real = tgt[:, 1:].to(device)   # 첫번째 <start> 토큰 제외

        # 디코더의 미래 토큰 참조 방지를 위한 마스크 생성
        tgt_mask = model.generate_square_subsequent_mask(tgt_input.size(1)).to(device)

        # 패딩 마스크의 경우 UserWarning 방지를 위해 명시적으로 bool 타입으로 변환
        src_padding_mask = (src == word_to_index["<pad>"]).to(device).bool()
        tgt_padding_mask = (tgt_input == word_to_index["<pad>"]).to(device).bool()

        # 기울기 초기화
        optimizer.zero_grad()

        # 모델 순전파 (Forward)
        output = model(src, tgt_input, tgt_mask=tgt_mask,
                       src_padding_mask=src_padding_mask, tgt_padding_mask=tgt_padding_mask)

        # 손실값 계산 (출력과 정답의 형태를 1차원으로 변경하여 비교)
        loss = criterion(output.view(-1, VOCAB_SIZE), tgt_real.contiguous().view(-1))

        # 역전파
        loss.backward()

        # 가중치 업데이트
        optimizer.step()

        # 스케줄러 업데이트
        scheduler.step()

        # 전체 손실값 누적
        total_loss += loss.item()

    # 10 에폭마다 진행 상황 출력
    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1}/{EPOCHS}, Loss: {total_loss/len(dataloader):.4f}")

# 디코딩 시 Beam Search 알고리즘
def evaluate_beam_search(sentence, model, beam_size=3, max_len=40):
    model.eval() # 모델을 평가 모드로 변경

    # 입력 문장 정제 및 토큰화 진행
    sentence = preprocess_sentence(sentence)
    tokens = mecab.morphs(sentence)
    seq = [word_to_index.get(word, word_to_index["<unk>"]) for word in tokens]
    src = torch.tensor([seq]).to(device)

    # Beam 탐색을 위한 초기값 설정 (누적 점수, 시퀀스 리스트)
    start_token = word_to_index["<start>"]
    beams = [(0.0, [start_token])]

    for i in range(max_len):
        new_beams = []

        for score, seq_list in beams:
            # 이미 <end> 토큰이 등장한 빔은 더 이상 확장하지 않고 유지
            if seq_list[-1] == word_to_index["<end>"]:
                new_beams.append((score, seq_list))
                continue

            # 현재까지 생성된 시퀀스를 텐서로 변환
            tgt = torch.tensor([seq_list]).to(device)

            # 마스크 생성 (bool 타입 적용)
            tgt_mask = model.generate_square_subsequent_mask(tgt.size(1)).to(device)
            src_padding_mask = (src == word_to_index["<pad>"]).to(device).bool()

            # 모델 예측 수행 (기울기 계산 비활성화)
            with torch.no_grad():
                output = model(src, tgt, tgt_mask=tgt_mask, src_padding_mask=src_padding_mask)

            # 마지막 단어의 확률 분포 추출 (Log Softmax 사용)
            prob = F.log_softmax(output[:, -1, :], dim=-1).squeeze(0)

            # 반복 출력을 방지하기 위한 페널티 부여
            if len(seq_list) > 1:
                last_word = seq_list[-1]
                prob[last_word] -= 1e4

            # 확률이 가장 높은 beam_size 개의 후보 단어 선택
            topk_prob, topk_idx = torch.topk(prob, beam_size)

            # 새로운 빔 조합
            for p, idx in zip(topk_prob, topk_idx):
                new_score = score + p.item()
                new_seq = seq_list + [idx.item()]
                new_beams.append((new_score, new_seq))

        # 점수 기준으로 정렬하여 상위 beam_size 개만 유지
        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]

        # 모든 빔이 <end>로 종료되었을 경우 반복문 탈출
        if all(seq[-1] == word_to_index["<end>"] for _, seq in beams):
            break

    # 최종적으로 가장 점수가 높은 빔 선택
    best_seq = beams[0][1]

    # <start> 및 <end> 토큰을 제외하고 문자열 변환
    decoded_words = []
    for idx in best_seq:
        if idx == word_to_index["<start>"]:
            continue
        if idx == word_to_index["<end>"]:
            break
        decoded_words.append(index_to_word[idx])

    return ' '.join(decoded_words)

# 형태소 분석기로 인해 분리된 조사와 기호들의 띄어쓰기를 교정하는 후처리 함수
def post_process_sentence(sentence):
    # 구두점 앞의 공백 제거
    sentence = sentence.replace(" .", ".")
    sentence = sentence.replace(" ?", "?")
    sentence = sentence.replace(" !", "!")
    sentence = sentence.replace(" ,", ",")

    # 잦은 빈도로 분리되는 조사 앞의 공백 제거
    particles = [' 이 ', ' 가 ', ' 은 ', ' 는 ', ' 을 ', ' 를 ', ' 도 ', ' 에 ', ' 에게 ', ' 에서 ', ' 로 ', ' 으로 ', ' 고 ', ' 라고 ', ' 다고 ', ' 에요 ', ' 예요 ', ' 죠 ', ' 습니다 ']
    for p in particles:
        sentence = sentence.replace(p, p.lstrip())

    return sentence.strip()

# 평가 및 후처리 파이프라인 함수
def evaluate_and_format(sentence, model, beam_size=3):
    raw_output = evaluate_beam_search(sentence, model, beam_size=beam_size)
    formatted_output = post_process_sentence(raw_output)
    return formatted_output

모델을 100 Epoch 만큼 학습
Epoch: 10/100, Loss: 2.1909
Epoch: 20/100, Loss: 1.0057
Epoch: 30/100, Loss: 0.6222
Epoch: 40/100, Loss: 0.4445
Epoch: 50/100, Loss: 0.3482
Epoch: 60/100, Loss: 0.2927
Epoch: 70/100, Loss: 0.2493
Epoch: 80/100, Loss: 0.2229
Epoch: 90/100, Loss: 0.1964
Epoch: 100/100, Loss: 0.1781


In [84]:
# 최종 학습 완료 후 결과 확인
print("\n[ 100 Epoch 학습 완료 후 결과 테스트 ]")
examples = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야."
]

for ex in examples:
    print(f"Q: {ex}\nA: {evaluate_and_format(ex, model)}\n")


[ 100 Epoch 학습 완료 후 결과 테스트 ]
Q: 지루하다, 놀러가고 싶어.
A: 저도 요!

Q: 오늘 일찍 일어났더니 피곤하다.
A: 아무래도 그렇 죠.

Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 상상은 해도 상관 없 어요.

Q: 집에 있는다는 소리야.
A: 참 행복 하 네요.



In [89]:
import re

# 형태소 분석기로 인해 분리된 한국어 띄어쓰기를 교정하는 향상된 후처리 함수
def post_process_sentence(sentence):
    # 1. 구두점 앞의 공백 제거 (예: "안녕 !" -> "안녕!")
    sentence = re.sub(r'\s+([?.!,])', r'\1', sentence)

    # 2. 한국어 형태소 분석 시 잦은 빈도로 분리되는 조사, 어미 리스트
    particles = [
        '이', '가', '은', '는', '을', '를', '도', '에', '에게', '에서', '로', '으로',
        '고', '라고', '다고', '요', '에요', '예요', '죠', '습니다', '네', '네요',
        '어', '어요', '아', '아요', '지', '잖아', '잖아요', '야', '다', '니까', '면'
    ]

    # 앞 글자가 한글일 경우, 리스트에 있는 조사/어미 앞의 공백을 제거 (예: "저 도" -> "저도")
    for p in particles:
        sentence = re.sub(r'(?<=[가-힣])\s+(?=' + p + r'(?:\s|$|[?.!,]))', '', sentence)

    # 3. 동사/형용사 어간(하, 되, 같, 없, 있 등) 뒤에 띄어쓰기가 발생한 경우 공백 제거 (예: "없 어요" -> "없어요")
    stems = ['하', '되', '같', '없', '있', '그렇', '어떻', '안', '않', '못']
    for stem in stems:
        sentence = re.sub(r'(?<=\b' + stem + r')\s+(?=[가-힣])', '', sentence)

    # 4. 명사와 '하다'가 분리된 경우 붙여쓰기 (예: "행복 하네요" -> "행복하네요")
    sentence = sentence.replace(" 하", "하")
    sentence = sentence.replace(" 해", "해")

    # 5. 연속된 다중 공백을 하나의 공백으로 치환
    sentence = re.sub(r'\s+', ' ', sentence)

    return sentence.strip()

# 평가 및 후처리 파이프라인 함수 재정의
def evaluate_and_format(sentence, model, beam_size=3):
    # Beam Search 알고리즘으로 모델 결과 예측
    raw_output = evaluate_beam_search(sentence, model, beam_size=beam_size)
    # 정규식을 적용한 향상된 띄어쓰기 후처리 실행
    formatted_output = post_process_sentence(raw_output)
    return formatted_output

# 수정된 후처리 함수를 기존 결과에 임시 테스트 (출력 형태 확인용)
test_sentences = [
    "저 도 요 !",
    "아무래도 그렇 죠 .",
    "상상은 해도 상관 없 어요 .",
    "참 행복 하 네요 ."
]

print("[ 개선된 띄어쓰기 후처리 함수 단독 테스트 ]")
for text in test_sentences:
    print(f"원래 출력: {text}\n교정 출력: {post_process_sentence(text)}\n")

# 최종 모델 테스트 루프
print("\n[ 100 Epoch 학습 완료 후 최종 결과 테스트 ]")
examples = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야."
]

for ex in examples:
    print(f"Q: {ex}\nA: {evaluate_and_format(ex, model, beam_size=3)}\n")

[ 개선된 띄어쓰기 후처리 함수 단독 테스트 ]
원래 출력: 저 도 요 !
교정 출력: 저도요!

원래 출력: 아무래도 그렇 죠 .
교정 출력: 아무래도 그렇죠.

원래 출력: 상상은 해도 상관 없 어요 .
교정 출력: 상상은해도 상관 없어요.

원래 출력: 참 행복 하 네요 .
교정 출력: 참 행복하네요.


[ 100 Epoch 학습 완료 후 최종 결과 테스트 ]
Q: 지루하다, 놀러가고 싶어.
A: 저도요!

Q: 오늘 일찍 일어났더니 피곤하다.
A: 아무래도 그렇죠.

Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 상상은해도 상관 없어요.

Q: 집에 있는다는 소리야.
A: 참 행복하네요.



In [92]:
from nltk.translate.bleu_score import sentence_bleu

def calculate_bleu(reference, candidate, weights=(0.25, 0.25, 0.25, 0.25)):
    # reference는 리스트의 리스트, candidate는 1차원 리스트 형태로 입력되어야 합니다.
    return sentence_bleu([reference], candidate, weights=weights)

print("[ BLEU 점수 측정 테스트 ]")

# 가이드라인에 제시된 예문과 목표 정답
test_pairs = [
    ("지루하다, 놀러가고 싶어.", "잠깐 쉬어도 돼요."),
    ("오늘 일찍 일어났더니 피곤하다.", "맛난 거 드세요."),
    ("간만에 여자친구랑 데이트 하기로 했어.", "떨리겠죠."),
    ("집에 있는다는 소리야.", "좋아하면 그럴 수 있어요.")
]

for query, real_answer in test_pairs:
    # 1. 모델을 통해 답변 예측
    model_prediction = evaluate_and_format(query, model, beam_size=3)

    # 2. 형태소 분석기를 이용해 실제 정답과 예측 답변을 토큰(단어) 단위로 분리
    ref_tokens = mecab.morphs(real_answer)
    cand_tokens = mecab.morphs(model_prediction)

    # 3. BLEU 점수 계산
    score = calculate_bleu(ref_tokens, cand_tokens)

    print(f"질문: {query}")
    print(f"실제 정답: {real_answer}")
    print(f"모델 예측: {model_prediction}")
    print(f"BLEU 점수: {score:.5f}\n")

[ BLEU 점수 측정 테스트 ]
질문: 지루하다, 놀러가고 싶어.
실제 정답: 잠깐 쉬어도 돼요.
모델 예측: 저도요!
BLEU 점수: 0.00000

질문: 오늘 일찍 일어났더니 피곤하다.
실제 정답: 맛난 거 드세요.
모델 예측: 아무래도 그렇죠.
BLEU 점수: 0.00000

질문: 간만에 여자친구랑 데이트 하기로 했어.
실제 정답: 떨리겠죠.
모델 예측: 상상은해도 상관 없어요.
BLEU 점수: 0.00000

질문: 집에 있는다는 소리야.
실제 정답: 좋아하면 그럴 수 있어요.
모델 예측: 참 행복하네요.
BLEU 점수: 0.00000



[회고]
테스트 결과 BLEU 점수가 0점이 나왔는데, 그 이유는 번역기와 달리 챗봇의 일상 대화는 하나의 질문에 여러 가지 정답이 존재할 수 있기 때문이라고 생각.
예를 들어 "피곤하다"는 질문에 "아무래도 그렇죠"라고 자연스럽게 대답했지만, 정답지인 "맛난 거 드세요"와 겹치는 단어가 하나도 없어서 BLEU 점수는 낮게 측정됨.
이를 통해 문장 구조와 단어가 정확히 일치해야만 점수가 오르는 BLEU 스코어는 챗봇 성능을 평가하기에는 한계가 있음을 알 수 있었음.